# Centralized adversarial training · multiclass

## Inputs and configuration

The seed identifies the run and the device determines where tensors are processed.

In [ ]:
from pathlib import Path

EXECUTION_DIR = Path.cwd().resolve()
ROOT_CANDIDATES = (
    EXECUTION_DIR,
    *EXECUTION_DIR.parents,
    EXECUTION_DIR / "uav-network-intrusion-detection",
)
ROOT = next(
    (
        candidato / "deep-learning-models"
        for candidato in ROOT_CANDIDATES
        if (candidato / "deep-learning-models" / "Normal+Attacks.csv").is_file()
    ),
    None,
)
if ROOT is None:
    ROOT = next(
        (candidato for candidato in ROOT_CANDIDATES if (candidato / "Normal+Attacks.csv").is_file()),
        None,
    )
if ROOT is None:
    raise FileNotFoundError(
        "Could not find deep-learning-models/Normal+Attacks.csv from the current directory."
    )

__file__ = str(ROOT / "adversarial_training_multiclass.ipynb")

## Data, training and metrics

The checkpoint stores parameters such as weights and biases. The class reconstructs the layer sequence that receives them.

In [ ]:
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset


ROOT = Path(__file__).resolve().parent

# ── Run configuration ──
# The seed identifies the random choices in this run. A comparison across methods also
# depends on using the same training and test examples.
SEED = 42
BATCH_SIZE = 256
EPOCHS = 50

# ── Evaluation scale ──
# Here ε is expressed in standardized feature units. For a column with standard deviation s,
# 0.05 in this space corresponds to 0.05 × s in the original units.
FGSM_EPSILON = 0.05
ADV_WEIGHT = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_MODEL_DIR = ROOT / "treated_multiclass_classification" / "models"
RESULTS_DIR = ROOT.parent / "adversarial-attacks" / "adversarial_results" / "adversarial_training_multiclass"
RESULT_MODEL_DIR = RESULTS_DIR / "models"


class MLPMulticlass(nn.Module):
    def __init__(self, input_dim, config, n_classes):
        super().__init__()
        hp = config["hiperparametros"]
        layers = []
        previous = input_dim
        activation = nn.ReLU if hp["ativacao"] == "relu" else nn.Tanh

        for index in range(hp["n_camadas"]):
            units = hp[f"unidades_{index}"]
            dropout = hp[f"dropout_{index}"]
            layers.extend((nn.Linear(previous, units), activation()))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            previous = units

        layers.append(nn.Linear(previous, n_classes))
        self.rede = nn.Sequential(*layers)

    def forward(self, inputs):
        return self.rede(inputs)


def load_configuration():
    hp_path = ROOT / "bayesian_tuning" / "best_hyperparameters_summary_multiclass.json"
    metrics_path = ROOT / "treated_multiclass_classification" / "training_metrics.json"
    checkpoint_path = BASE_MODEL_DIR / "MLP_multiclass.pt"
    scaler_path = BASE_MODEL_DIR / "scaler_multiclass.joblib"
    encoder_path = BASE_MODEL_DIR / "label_encoder_multiclass.joblib"

    for path in (hp_path, metrics_path, checkpoint_path, scaler_path, encoder_path):
        if not path.is_file():
            raise FileNotFoundError(f"Required artifact not found: {path}")

    with hp_path.open(encoding="utf-8") as file:
        hyperparameters = json.load(file)["MLP"]
    with metrics_path.open(encoding="utf-8") as file:
        reference_metrics = json.load(file)["MLP"]

    return (
        hyperparameters,
        reference_metrics,
        checkpoint_path,
        scaler_path,
        joblib.load(encoder_path),
    )

Groups are keyed by feature values. Two records with identical vectors stay on the same side of the split even when their labels disagree. This controls one source of leakage. Evaluating generalization to a new capture requires capture identifiers. The cleaned task merges Reco and Scanning.

In [ ]:
def prepare_data(encoder):
    dataset_path = ROOT / "Normal+Attacks.csv"
    if not dataset_path.is_file():
        raise FileNotFoundError(f"Dataset not found: {dataset_path}")

    df = pd.read_csv(dataset_path)
    if "label" not in df:
        raise ValueError(f"The dataset is missing the required column 'label': {dataset_path}")

    df["label"] = df["label"].replace(
        {"Reco": "Reco+Scanning", "Scanning": "Reco+Scanning"}
    )
    if "fwd_seg_size_min" not in df or df["fwd_seg_size_min"].nunique(dropna=False) != 1:
        raise ValueError("The fwd_seg_size_min column does not match the saved cleaned-data protocol.")
    df = df.drop(columns=["fwd_seg_size_min"])
    features = df.drop(columns=["label"]).select_dtypes(include="number")
    if features.empty:
        raise ValueError("The dataset has no numeric features for training.")

    data = features.copy()
    data["target"] = df["label"].astype(str).to_numpy()

    # ── Duplicate rows ──
    # The target is already in the table. Deduplication compares complete records. Grouping
    # later handles identical feature vectors retained because their labels differ.
    data = data.drop_duplicates().reset_index(drop=True)
    X = data[features.columns]
    target_names = encoder.classes_.tolist()

    # ── Class mapping ──
    # Each output index comes from the encoder used during training. An unknown label would
    # break the interpretation of checkpoint predictions.
    unknown_labels = set(data["target"]) - set(target_names)
    if unknown_labels:
        raise ValueError(f"Labels with no match in the saved encoder: {sorted(unknown_labels)}")
    y = encoder.transform(data["target"]).astype("int64")

    # ── Groups for splitting ──
    # The hash depends on features and ignores the row index. Copies at different positions
    # receive the same group key.
    groups = pd.util.hash_pandas_object(X, index=False).to_numpy()
    split = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
    train_indices, test_indices = next(split.split(X, y, groups=groups))

    scaler = StandardScaler()

    # ── Scale fitted on training data ──
    # Each column has its own mean and scale. The subsequent transform reuses these values
    # even when the test distribution differs.
    X_train = scaler.fit_transform(X.iloc[train_indices]).astype("float32")
    X_test = scaler.transform(X.iloc[test_indices]).astype("float32")
    y_train = y[train_indices].astype("int64")
    y_test = y[test_indices].astype("int64")

    return (
        X_train,
        y_train,
        X_test,
        y_test,
        scaler,
        features.columns.tolist(),
        target_names,
    )



# The saved configuration specifies the architecture. The checkpoint supplies weights
# learned by the reference model, which is evaluated on the same test rows.
def load_model(checkpoint_path, hyperparameters, input_dim, class_names):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
    if int(checkpoint["n_features"]) != input_dim:
        raise ValueError("The number of checkpoint features does not match the prepared dataset.")
    if int(checkpoint["n_classes"]) != len(class_names):
        raise ValueError("The number of checkpoint classes does not match the saved encoder.")
    if checkpoint["class_names"] != class_names:
        raise ValueError("The class order in the checkpoint differs from the saved encoder order.")

    model = MLPMulticlass(input_dim, hyperparameters, len(class_names)).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    return model


def generate_fgsm(model, inputs, targets):
    original = inputs.detach()
    adversarial_input = original.clone().requires_grad_(True)
    loss = nn.functional.cross_entropy(model(adversarial_input), targets)
    gradient = torch.autograd.grad(loss, adversarial_input)[0]
    return (original + FGSM_EPSILON * gradient.sign()).detach()


def generate_test_fgsm(model, X_test, y_test):
    adversarial_batches = []
    model.eval()
    for start in range(0, len(X_test), BATCH_SIZE):
        inputs = torch.as_tensor(
            X_test[start : start + BATCH_SIZE], dtype=torch.float32, device=DEVICE
        )
        targets = torch.as_tensor(
            y_test[start : start + BATCH_SIZE], dtype=torch.int64, device=DEVICE
        )
        adversarial_batches.append(generate_fgsm(model, inputs, targets).cpu().numpy())
    return np.concatenate(adversarial_batches, axis=0)

The defense computes two losses against the same labels, one for clean and one for perturbed data. ADV_WEIGHT determines their contributions to the MLP update.

In [ ]:
def predict(model, inputs):
    model.eval()
    predictions = []
    with torch.no_grad():
        for start in range(0, len(inputs), 1024):
            batch = torch.as_tensor(
                inputs[start : start + 1024], dtype=torch.float32, device=DEVICE
            )
            predictions.extend(model(batch).argmax(dim=1).cpu().tolist())
    return np.asarray(predictions, dtype="int64")


def evaluate_metrics(y_true, y_pred, class_names, y_clean_pred=None):
    labels = np.arange(len(class_names))
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "confusion_matrix": matrix.tolist(),
        "classification_report": classification_report(
            y_true,
            y_pred,
            labels=labels,
            target_names=class_names,
            output_dict=True,
            zero_division=0,
        ),
    }

    if y_clean_pred is not None:

        # ── ASR denominator ──
        # Only examples this model got right on clean inputs are eligible. Ten new errors
        # among 80 initially correct predictions give 12.5% ASR.
        initially_correct = y_clean_pred == y_true
        denominator = int(initially_correct.sum())
        metrics["attack_success_rate"] = (
            float(np.logical_and(initially_correct, y_pred != y_true).sum() / denominator)
            if denominator
            else None
        )
        metrics["attack_success_definition"] = (
            "Fraction of clean-correct test samples misclassified after FGSM."
        )
    else:
        metrics["attack_success_rate"] = None

    return metrics



# The defended MLP has its own training run. Time spent generating perturbed batches is
# included in this function’s total training time.
def train_with_fgsm(X_train, y_train, hyperparameters, input_dim, n_classes):
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    model = MLPMulticlass(input_dim, hyperparameters, n_classes).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=hyperparameters["hiperparametros"]["learning_rate"]
    )
    criterion = nn.CrossEntropyLoss()
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(
        TensorDataset(
            torch.as_tensor(X_train, dtype=torch.float32),
            torch.as_tensor(y_train, dtype=torch.int64),
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
    )

    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
    total_start = time.perf_counter()
    fgsm_generation_seconds = 0.0
    for epoch in range(EPOCHS):
        model.train()
        for clean_inputs, targets in loader:
            clean_inputs = clean_inputs.to(DEVICE)
            targets = targets.to(DEVICE)

            model.eval()
            if DEVICE.type == "cuda":
                torch.cuda.synchronize(DEVICE)
            fgsm_start = time.perf_counter()
            adversarial_inputs = generate_fgsm(model, clean_inputs, targets)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize(DEVICE)
            fgsm_generation_seconds += time.perf_counter() - fgsm_start

            model.train()
            optimizer.zero_grad()
            clean_loss = criterion(model(clean_inputs), targets)
            adversarial_loss = criterion(model(adversarial_inputs), targets)

            # ── Batch objective ──
            # Both losses use the same targets. With ADV_WEIGHT = 0.5 each accounts for half
            # the loss before any FedProx penalty.
            loss = (1 - ADV_WEIGHT) * clean_loss + ADV_WEIGHT * adversarial_loss
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch + 1}/{EPOCHS} completed.")

    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
    training_seconds = time.perf_counter() - total_start
    return model, training_seconds, fgsm_generation_seconds

Accuracy counts all correct predictions. Macro-F1 gives equal weight to each class F1. For ASR, align clean and perturbed predictions from the same model by row. If it originally got 80 examples right and now misses 8 of those, ASR is 8/80 = 10%.

In [ ]:
def main():
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    (
        hyperparameters,
        reference_metrics,
        checkpoint_path,
        scaler_path,
        encoder,
    ) = load_configuration()
    X_train, y_train, X_test, y_test, scaler, feature_names, class_names = prepare_data(encoder)


    # The saved scaler helps check the reconstructed split. Different means or scales may
    # point to changed rows or column order.
    saved_scaler = joblib.load(scaler_path)
    if not np.allclose(saved_scaler.mean_, scaler.mean_) or not np.allclose(
        saved_scaler.scale_, scaler.scale_
    ):
        raise ValueError("The reconstructed scaler does not match the saved scaler.")

    baseline = load_model(checkpoint_path, hyperparameters, X_train.shape[1], class_names)
    clean_baseline_predictions = predict(baseline, X_test)
    baseline_clean_metrics = evaluate_metrics(y_test, clean_baseline_predictions, class_names)
    for metric_name in ("accuracy", "macro_f1"):
        if not np.isclose(
            baseline_clean_metrics[metric_name], float(reference_metrics[metric_name]), atol=1e-12
        ):
            raise ValueError(
                f"Checkpoint metric {metric_name} does not match "
                "training_metrics.json. Check that the cleaned multiclass artifacts are consistent."
            )

    with torch.enable_grad():
        baseline_adversarial_test = generate_test_fgsm(baseline, X_test, y_test)
    baseline_attacked_predictions = predict(baseline, baseline_adversarial_test)

    adv_model, training_seconds, fgsm_generation_seconds = train_with_fgsm(
        X_train, y_train, hyperparameters, X_train.shape[1], len(class_names)
    )
    clean_adv_predictions = predict(adv_model, X_test)
    with torch.enable_grad():
        adv_adversarial_test = generate_test_fgsm(adv_model, X_test, y_test)
    attacked_adv_predictions = predict(adv_model, adv_adversarial_test)

    # ── Evaluation conditions ──
    # Clean and perturbed predictions are paired by example. Their difference measures the
    # sensitivity of the evaluated model.
    conditions = {
        "NoAdv_clean_test": evaluate_metrics(y_test, clean_baseline_predictions, class_names),
        "NoAdv_FGSM_test": evaluate_metrics(
            y_test, baseline_attacked_predictions, class_names, clean_baseline_predictions
        ),
        "AdvTrain_clean_test": evaluate_metrics(y_test, clean_adv_predictions, class_names),
        "AdvTrain_FGSM_test": evaluate_metrics(
            y_test, attacked_adv_predictions, class_names, clean_adv_predictions
        ),
    }

    def relative_recovery(clean_accuracy, attacked_accuracy, defended_accuracy):
        impact = clean_accuracy - attacked_accuracy
        return (defended_accuracy - attacked_accuracy) / impact if impact > 0 else None

    accuracy_recovery = relative_recovery(
        conditions["NoAdv_clean_test"]["accuracy"],
        conditions["NoAdv_FGSM_test"]["accuracy"],
        conditions["AdvTrain_FGSM_test"]["accuracy"],
    )
    macro_f1_recovery = relative_recovery(
        conditions["NoAdv_clean_test"]["macro_f1"],
        conditions["NoAdv_FGSM_test"]["macro_f1"],
        conditions["AdvTrain_FGSM_test"]["macro_f1"],
    )

    RESULT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "state_dict": adv_model.state_dict(),
            "n_classes": len(class_names),
            "n_features": len(feature_names),
            "class_names": class_names,
            "feature_names": feature_names,
        },
        RESULT_MODEL_DIR / "MLP_FGSM_multiclass.pt",
    )
    joblib.dump(scaler, RESULT_MODEL_DIR / "scaler_multiclass.joblib")
    joblib.dump(encoder, RESULT_MODEL_DIR / "label_encoder_multiclass.joblib")

    results = {
        "configuration": {
            "dataset": "Normal+Attacks.csv",
            "scenario": "multiclass_cleaned",
            "seed": SEED,
            "split": "StratifiedGroupKFold: fold 1 de 4; grupos por vetor de features",
            "training_samples": int(len(y_train)),
            "test_samples": int(len(y_test)),
            "test_class_counts": {
                name: int((y_test == index).sum()) for index, name in enumerate(class_names)
            },
            "classes": class_names,
            "model": "MLP",
            "training_and_test_attack": "Untargeted FGSM",
            "epsilon": FGSM_EPSILON,
            "epsilon_unit": "standardized feature space (StandardScaler)",
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "peso_adversarial": ADV_WEIGHT,
            "device": str(DEVICE),
        },
        "metrics": conditions,
        "relative_accuracy_recovery": accuracy_recovery,
        "relative_macro_f1_recovery": macro_f1_recovery,
        "cost": {
            "adversarial_training_seconds": training_seconds,
            "fgsm_generation_during_training_seconds": fgsm_generation_seconds,
            "note": (
                "NoAdv training time was not measured on the same device; it is not possible to "
                "estimate relative training overhead."
            ),
        },
    }
    output_path = RESULTS_DIR / "comparative_metrics.json"
    output_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Results saved to: {output_path}")
    print(f"Relative accuracy recovery: {accuracy_recovery}")
    print(f"Relative macro-F1 recovery: {macro_f1_recovery}")
    for condition, metrics in conditions.items():
        print(
            f"{condition}: accuracy={metrics['accuracy']:.4f}, "
            f"macro_f1={metrics['macro_f1']:.4f}, "
            f"attack_success_rate={metrics['attack_success_rate']}"
        )

## Output directory

Metrics, model and preprocessing belong together. When reusing a checkpoint, load its matching scaler and, for multiclass tasks, check the label order in the encoder.

In [ ]:
RESULTS_DIR = (
    ROOT.parent
    / "adversarial-attacks"
    / "adversarial_results"
    / "adversarial_training_multiclass"
)
RESULT_MODEL_DIR = RESULTS_DIR / "models"

## Run the experiment

The routine prepares the data, evaluates the reference, trains and saves results. A full run can overwrite files for the same experiment.

In [ ]:
main()